**Superdense coding.**
Superdense coding (Bennett and Wiesner, 1992) allows two classical bits
to be communicated by transmitting only **one qubit**, provided the sender
and receiver share a pre-entangled Bell pair.
It is the converse of quantum teleportation: teleportation sends one qubit
using two classical bits; superdense coding sends two classical bits using
one qubit.

The protocol has three phases:

1.) **Bell pair creation (shared resource).**
A Bell pair $\lvert\Phi^+\rangle = \frac{1}{\sqrt{2}}(\lvert 00\rangle + \lvert 11\rangle)$
is created and one qubit (q0) is given to the sender (Alice),
the other (q1) to the receiver (Bob).
This entanglement is established in advance - Alice and Bob can be
arbitrarily far apart when the actual message is sent.

2.) **Encoding (Alice).**
Alice applies one of four single-qubit operations to her qubit q0
to encode her two-bit message $(a, b)$:

| $a$ | $b$ | Gate on q0 | Resulting Bell state |
|---|---|---|---|
| 0 | 0 | $I$ | $\frac{1}{\sqrt{2}}(\lvert 00\rangle + \lvert 11\rangle) = \lvert\Phi^+\rangle$ |
| 0 | 1 | $Z$ | $\frac{1}{\sqrt{2}}(\lvert 00\rangle - \lvert 11\rangle) = \lvert\Phi^-\rangle$ |
| 1 | 0 | $X$ | $\frac{1}{\sqrt{2}}(\lvert 10\rangle + \lvert 01\rangle) = \lvert\Psi^+\rangle$ |
| 1 | 1 | $ZX$ | $\frac{1}{\sqrt{2}}(\lvert 01\rangle - \lvert 10\rangle) = \lvert\Psi^-\rangle$ |

<span style="color: red;">**Alice then sends her <u>single</u> qubit q0 to Bob.**</span>


3.) **Decoding (Bob).**
Bob applies CNOT(q0 $\rightarrow$ q1) then H(q0) to disentangle the Bell state
and recover the two classical bits.
Measuring q0 gives $a$ and measuring q1 gives $b$.

---
**Cell 01 - `superdense_coding()` function and shared backend.**
The function takes two classical bits $a$ and $b$, constructs the
full circuit, displays the circuit diagram, the three intermediate
statevectors (after Bell pair creation, after Alice's encoding,
after Bob's decoding gates), and the measurement histogram.

In [ ]:
"""superdense_coding.ipynb"""

# Cell 01 - Superdense coding: encode two classical bits into one qubit

import qiskit
from IPython.display import Math, display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

from qis101_utils import as_latex

backend = AerSimulator()


def superdense_coding(a: int, b: int) -> None:
    """Demonstrate superdense coding for a 2-bit message (a, b).

    Constructs and runs the full superdense coding circuit:
    Bell pair creation, Alice's encoding of (a, b) onto q0,
    and Bob's decoding via CNOT and H.
    Displays the circuit, three statevectors, and measurement histogram.

    Parameters
    ----------
    a, b : int
        Classical bits to encode (each 0 or 1).
        Measurement of q0 recovers a; measurement of q1 recovers b.
    """
    qc = QuantumCircuit(2, 2)

    # --- Bell pair creation ---
    # H on q0 then CNOT(q0->q1): creates |Phi+> = (|00>+|11>)/sqrt2
    qc.h(0)
    qc.cx(0, 1)
    qc.save_statevector("sv1")  # sv1: Bell pair |Phi+>

    # --- Alice's encoding ---
    # Apply one of four single-qubit operations to q0 based on (a, b)
    if a == 0 and b == 0:
        qc.id(0)  # I  -> |Phi+>: (|00>+|11>)/sqrt2
    if b == 1:
        qc.z(0)  # Z  -> |Phi->: (|00>-|11>)/sqrt2
    if a == 1:
        qc.x(0)  # X  -> |Psi+>: (|01>+|10>)/sqrt2  (b=0)
        # ZX -> |Psi->: (|01>-|10>)/sqrt2  (b=1)
    qc.save_statevector("sv2")  # sv2: encoded Bell state

    # --- Bob's decoding ---
    # CNOT(q0->q1) then H(q0) disentangles the Bell state
    # and maps it back to a computational basis state |ab>
    qc.cx(0, 1)
    qc.h(0)
    qc.save_statevector("sv3")  # sv3: decoded state |ab>

    # Measure q0 -> classical bit 0 (recovers a)
    # Measure q1 -> classical bit 1 (recovers b)
    qc.measure(0, 0)
    qc.measure(1, 1)

    result = backend.run(transpile(qc, backend)).result()

    sv1 = result.data(0)["sv1"]
    sv2 = result.data(0)["sv2"]
    sv3 = result.data(0)["sv3"]

    display(Math(rf"\large\mathbf{{a={a},\;b={b}}}"))
    display(qc.draw(output="mpl"))
    display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
    display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
    display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
    display(plot_distribution(result.get_counts(qc)))


print(f"Qiskit SDK Version: {qiskit.__version__}")
print(f"Backend name     : {backend.name}")
print(f"Aer version      : {backend.configuration().backend_version}")

**Cell 02 - Encoding message $(a=0, b=0)$.**
Both bits are 0. Alice applies the identity gate $I$ to q0,
leaving the Bell pair unchanged as $\lvert\Phi^+\rangle$.
The histogram shows `00` with probability 1.

In [ ]:
# Message: a=0, b=0
superdense_coding(0, 0)


**Cell 03 - Encoding message $(a=0, b=1)$.**
Alice applies $Z$ to q0, transforming the Bell pair to
$\lvert\Phi^-\rangle = \frac{1}{\sqrt{2}}(\lvert 00\rangle - \lvert 11\rangle)$.
The histogram shows `01` with probability 1.

In [ ]:
# Message: a=0, b=1
superdense_coding(0, 1)


**Cell 04 - Encoding message $(a=1, b=0)$.**
Alice applies $X$ to q0, transforming the Bell pair to
$\lvert\Psi^+\rangle = \frac{1}{\sqrt{2}}(\lvert 10\rangle + \lvert 01\rangle)$.
The histogram shows `10` with probability 1.

In [ ]:
# Message: a=1, b=0
superdense_coding(1, 0)


**Cell 05 - Encoding message $(a=1, b=1)$.**
Alice applies $ZX$ to q0, transforming the Bell pair to
$\lvert\Psi^-\rangle = \frac{1}{\sqrt{2}}(\lvert 01\rangle - \lvert 10\rangle)$.
The histogram shows `11` with probability 1.

In [ ]:
# Message: a=1, b=1
superdense_coding(1, 1)
